In [10]:
df = pd.read_csv("../lesson-1-3-summary-statistics/skyline_enrollments.csv")

In [11]:
copy ..\lesson-1-3-summary-statistics\skyline_enrollments.csv .

        1 file(s) copied.


In [12]:
import os
print(os.getcwd())
print(os.path.exists("../lesson-1-3-summary-statistics/skyline_enrollments.csv"))

c:\Users\nasib_vr6iaim\northstar-coursework\module-01-foundations-of-analytics-and-statistics\lesson-1-7-correlation-and-ab-testing
True


In [13]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

df = pd.read_csv("skyline_enrollments.csv")
df["grade_num"] = df["final_grade"].map({"F": 0, "D": 1, "C": 2, "B": 3, "A": 4})

r, p_r = pearsonr(df["hours_studied"], df["grade_num"])
rho, p_s = spearmanr(df["hours_studied"], df["grade_num"])
print(f"Pearson r = {r:.3f} (p = {p_r:.3f})")
print(f"Spearman rho = {rho:.3f} (p = {p_s:.3f})")

Pearson r = 0.070 (p = 0.488)
Spearman rho = 0.092 (p = 0.364)


### Possible confounders for `hours_studied` and `final_grade`

1. **Prior ability or background knowledge.** Students who already understand the material may earn high grades with little studying, while struggling students may study heavily and still score lower. Ability affects both variables, so it can distort the correlation in either direction even if studying itself has no effect on learning. This isn't measured in our data.

2. **Motivation and conscientiousness.** Highly motivated students tend to study more and also do the other things that raise grades, such as attending class, turning in assignments, and asking for help. The grade boost could come from those habits rather than from the extra study hours.

3. **Course difficulty (`course_name`).** Harder courses may demand more study hours and also have tougher grading, which pushes hours up and grades down at the same time. If the courses differ in difficulty and the correlation is computed across all of them, that can hide or flip the real relationship. This one is in our data, so we can check it by correlating within each course.

4. **Outside commitments (related to `age`).** Older students may balance jobs or family, leaving fewer hours to study, while also bringing more life experience or focus that helps their grades. Age is available in our data, so it can be controlled for.

## A/B Test Design: Does the Study Planner Cause Higher Grades?

### 1. Treatment and control
- **Treatment:** the study planner is switched on. It automatically schedules study sessions for the student.
- **Control:** the planner is off and invisible, with everything else about the platform identical. Control students don't get a different feature or a placebo notification, so the only difference is the planner.

### 2. Randomization unit: per student
Randomize individual students, not enrollments or cohorts.
- **Not per-enrollment:** one student can be in several courses, and the same student would then be in both arms, so the planner's effect would leak into the control group.
- **Not per-cohort:** with only a handful of cohorts, the sample would be tiny, and cohort differences (instructor, term, course mix) would be tangled up with the treatment.
- **Stratify by `course_name`** so each course has a balanced split of treatment and control.
- Note: the current data has no `student_id`, so we would need to add one to randomize and analyze at the student level.

### 3. Outcome metric
- **Primary:** whether the student finishes with a final grade of **B or higher** (binary).
- **Analysis:** intent-to-treat, meaning students are compared by the arm they were *assigned* to, whether or not they used the planner. This protects the randomization.
- **Withdrawals:** students who drop the course count as "not B or higher," decided before the test starts.
- **Secondary (explanatory only):** planner usage rate and `hours_studied`, to see whether any effect works through study time.
- **Guardrails:** withdrawal rate and student satisfaction, so a grade gain isn't bought with a worse experience.

### 4. Minimum meaningful effect: +5 percentage points
Lifting B-or-higher from 40% to 45% (about one extra strong student for every 20 enrolled) is, in my judgment, roughly the smallest gain that would justify the engineering and maintenance cost of the feature. A smaller target is expensive to detect: sample size grows with the inverse square of the effect, so a 3-point lift would need roughly 4,000+ per group.

### 5. Sample size (α = 0.05, power = 80%, two-sided)

| Baseline | Target | Per group | Total |
|---|---|---|---|
| 40% (assumed) | 45% | **1,534** | **3,068** |
| 56% (actual in our data) | 61% | 1,524 | 3,048 |

Plan for roughly **3,100 students total** (about 1,550 per arm), plus a buffer for withdrawals or missing grades. For reference, a 10-point lift (40% to 50%) would need only about 388 per group.

**Practical note:** our sample has only 100 enrollments over five years, so reaching about 3,100 students would take a long time at current volume. The team should either run the test for multiple terms or decide whether a larger minimum effect is acceptable.

In [14]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

for p1, p2 in [(0.40, 0.45), (0.56, 0.61)]:
    es = proportion_effectsize(p2, p1)
    n = NormalIndPower().solve_power(effect_size=es, alpha=0.05, power=0.80, ratio=1)
    print(f"{p1:.0%} -> {p2:.0%}: {round(n)} per group, {2*round(n)} total")

40% -> 45%: 1533 per group, 3066 total
56% -> 61%: 1523 per group, 3046 total
